# 05b. (Optional) English translations for display

Translates the Portuguese reviews to English with a free local model so the demo is readable; it is display-only.

## What to look at

- Original Portuguese next to the English translation.
- **Nothing else depends on this step**: embeddings, the index and all benchmarks use the original text.
- It is slow on a laptop. Set `RUN_TRANSLATION = False` to skip it (the cell then only shows an existing table).

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
RUN_TRANSLATION = True   # False = skip the slow step
if RUN_TRANSLATION:
    run_script("05b_translate_reviews.py")
else:
    print("skipped")

In [ ]:
spark = get_spark("nb05b")
if spark.catalog.tableExists(T_TRANS):
    display(spark.table(T_TRANS).where("length(review_text) > 40").limit(8).toPandas())
    total = spark.table(T_TRANS).count()
    missing = spark.table(T_SILVER).select("review_text").distinct().join(spark.table(T_TRANS), "review_text", "left_anti").count()
    print("translated texts:", total, "| silver texts without a translation:", missing)
else:
    print("no translation table (optional step not run)")

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

If the step ran: new folder `silver/review_translations/` with **metadata/** and **data/**, where many tiny Parquet files appear (one per Spark task). If you skipped it, nothing changes.